In [0]:
%sql
DROP TABLE IF EXISTS inlap.control.audit_log;
DROP TABLE IF EXISTS inlap.control.watermark_table;

CREATE TABLE inlap.control.audit_log (
    source_vendor STRING,
    layer STRING,
    run_timestamp TIMESTAMP,
    status STRING,
    rows_read LONG,
    rows_passed LONG,
    rows_quarantined LONG
) USING DELTA;

CREATE TABLE inlap.control.watermark_table (
    source_vendor STRING,
    last_watermark_value TIMESTAMP,
    last_run_timestamp TIMESTAMP,
    status STRING
) USING DELTA;

In [0]:
spark.sql("""
    INSERT INTO inlap.control.watermark_table (source_vendor, last_watermark_value, last_run_timestamp, status)
    VALUES ('here_tech', '2020-01-01 00:00:00', current_timestamp(), 'INITIALIZED')
""")

In [0]:
spark.sql("""
    INSERT INTO inlap.control.watermark_table (source_vendor, last_watermark_value, last_run_timestamp, status)
    VALUES ('georesults', '2020-01-01 00:00:00', current_timestamp(), 'INITIALIZED')
""")

In [0]:
spark.sql("""
    INSERT INTO inlap.control.watermark_table (source_vendor, last_watermark_value, last_run_timestamp, status)
    VALUES ('nv5', '2020-01-01 00:00:00', current_timestamp(), 'INITIALIZED')
""")

In [0]:
spark.sql("""
    INSERT INTO inlap.control.watermark_table (source_vendor, last_watermark_value, last_run_timestamp, status)
    VALUES ('opensignal', '2020-01-01 00:00:00', current_timestamp(), 'INITIALIZED')
""")

In [0]:
spark.sql("""
    INSERT INTO inlap.control.watermark_table (source_vendor, last_watermark_value, last_run_timestamp, status)
    VALUES ('precisely', '2020-01-01 00:00:00', current_timestamp(), 'INITIALIZED')
""")

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS inlap.silver;

-- Drop any broken managed silver tables before recreating as external
DROP TABLE IF EXISTS inlap.silver.heretech_conformed;
DROP TABLE IF EXISTS inlap.silver.georesults_conformed;
DROP TABLE IF EXISTS inlap.silver.nv5_conformed;
DROP TABLE IF EXISTS inlap.silver.opensignal_conformed;
DROP TABLE IF EXISTS inlap.silver.precisely_conformed;
DROP TABLE IF EXISTS inlap.silver.quarantine_records;
-- All vendor-conformed silver tables as external Delta under ADLS silver folder
CREATE TABLE IF NOT EXISTS inlap.silver.heretech_conformed (
    site_id STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    street_address STRING,
    city STRING,
    state STRING,
    network_type STRING,
    signal_strength_dbm DOUBLE
) USING DELTA
LOCATION 'abfss://datalake@attinlapsa.dfs.core.windows.net/silver/heretech_conformed/';

CREATE TABLE IF NOT EXISTS inlap.silver.georesults_conformed (
    record_id INT,
    site_name STRING,
    address1 STRING,
    address2 STRING,
    municipality STRING,
    postal_code INT,
    confidence_score DOUBLE,
    last_touch TIMESTAMP,
    _ingestion_timestamp TIMESTAMP,
    source_file STRING,
    source_name STRING,
    lat DOUBLE,
    lon DOUBLE,
    site_id STRING,
    state STRING
) USING DELTA
LOCATION 'abfss://datalake@attinlapsa.dfs.core.windows.net/silver/georesults_conformed/';

CREATE TABLE IF NOT EXISTS inlap.silver.nv5_conformed (
    site_id STRING,
    lat DOUBLE,
    lon DOUBLE,
    street_address STRING,
    city STRING,
    state STRING
) USING DELTA
LOCATION 'abfss://datalake@attinlapsa.dfs.core.windows.net/silver/nv5_conformed/';

CREATE TABLE IF NOT EXISTS inlap.silver.opensignal_conformed (
    site_ref STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    network_type STRING
) USING DELTA
LOCATION 'abfss://datalake@attinlapsa.dfs.core.windows.net/silver/opensignal_conformed/';

CREATE TABLE IF NOT EXISTS inlap.silver.precisely_conformed (
    LOC_ID STRING,
    ADDR_FULL STRING,
    city STRING,
    state_cd STRING,
    zip STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    VALIDATION_STATUS STRING
) USING DELTA
LOCATION 'abfss://datalake@attinlapsa.dfs.core.windows.net/silver/precisely_conformed/';

CREATE TABLE IF NOT EXISTS inlap.silver.quarantine_records (
    raw_record STRING,
    failure_reason STRING,
    source_vendor STRING,
    quarantine_timestamp TIMESTAMP
) USING DELTA
LOCATION 'abfss://datalake@attinlapsa.dfs.core.windows.net/silver/quarantine_records/';

In [0]:
spark.sql("""
    INSERT INTO inlap.control.watermark_table (source_vendor, last_watermark_value, last_run_timestamp, status)
    SELECT 'geolink', CAST('2020-01-01 00:00:00' AS TIMESTAMP), current_timestamp(), 'INITIALIZED'
    WHERE NOT EXISTS (
        SELECT 1
        FROM inlap.control.watermark_table
        WHERE source_vendor = 'geolink'
    )
""")

In [0]:
%sql
DROP TABLE IF EXISTS inlap.silver.geolink_conformed;
DROP TABLE IF EXISTS inlap.silver.sites_unified;

CREATE TABLE IF NOT EXISTS inlap.silver.geolink_conformed (
    baseglid STRING,
    glid STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    full_address STRING,
    unit_number STRING,
    city STRING,
    state STRING,
    zip STRING,
    location_type STRING,
    confidence_score DOUBLE,
    last_verified DATE,
    lat_rounded DOUBLE,
    lon_rounded DOUBLE,
    location_glid STRING
) USING DELTA
LOCATION 'abfss://datalake@attinlapsa.dfs.core.windows.net/silver/geolink_conformed/';

CREATE TABLE IF NOT EXISTS inlap.silver.sites_unified (
    site_id STRING,
    street_address STRING,
    city STRING,
    state STRING,
    zip STRING,
    lat DOUBLE,
    lon DOUBLE,
    network_type STRING,
    status STRING,
    source_vendor STRING,
    lat_rounded DOUBLE,
    lon_rounded DOUBLE,
    matched_baseglid_count BIGINT,
    matched_glid_count BIGINT,
    latest_registry_verification DATE,
    max_registry_confidence DOUBLE,
    representative_baseglid STRING,
    representative_glid STRING,
    resolved_street_address STRING,
    resolved_unit_number STRING,
    resolved_city STRING,
    resolved_state STRING,
    resolved_zip STRING,
    resolved_location_type STRING,
    registry_confidence_score DOUBLE,
    registry_last_verified DATE,
    geolink_match_status STRING,
    resolved_baseglid STRING,
    resolved_glid STRING,
    registry_join_grain STRING,
    canonical_street_address STRING,
    canonical_city STRING,
    canonical_state STRING,
    canonical_zip STRING,
    master_site_id STRING
) USING DELTA
LOCATION 'abfss://datalake@attinlapsa.dfs.core.windows.net/silver/sites_unified/';